<div style="text-align:right"><i>code and commentary by Claude Fable 5<br>July 2026</i></div>

# Project Euler 1–100: Claude Fable 5 Edition

This notebook is a companion to [Euler.ipynb](Euler.ipynb) (Peter Norvig's solutions, which include commentary) and
[Euler-Opus.ipynb](Euler-Opus.ipynb) (an  AI edition by Claude Opus 4.8). The brief for this edition, written by Claude Fable 5: solve
Project Euler problems 1–100 with **every solution running in under one second**. The whole
notebook runs in about two seconds. Getting there is mostly about algorithms, not
micro-optimization:

- **Sieves over tests**: bulk primality and divisor-sum questions use numpy sieves; individual
  queries use deterministic Miller-Rabin.
- **Count classes, not instances**: several problems (74, 92) group the numbers below ten
  million by digit multiset, shrinking 10,000,000 cases to about 10,000.
- **Exact math over search or simulation**: closed forms (1, 6, 28), Pell equations (66, 94),
  a Markov chain rather than a Monte Carlo simulation (84), and the totient summatory
  recurrence (72).
- **Prune the search space first**: mod-3 classes cut the prime-pair graph in half (60);
  divisibility chains the pandigital construction (43).

Each solution is a function `euler_N()` that returns the answer; `run(euler_N)` checks it
against the expected answer and records the run time. Data files and the harness live in
[euler_data.py](euler_data.py).

In [1]:
from euler_data import DATA, run, runs

## Setup

In [2]:
"""Utility functions shared by the solutions.

The goal throughout is speed: every problem should run in well under a
second. Numpy handles sieves and other bulk array work; a deterministic
Miller-Rabin test handles individual primality queries.
"""

from collections import Counter, defaultdict
from datetime import date
from fractions import Fraction
from functools import lru_cache
from itertools import combinations, combinations_with_replacement, count, permutations
from math import comb, factorial, gcd, inf, isqrt, lcm, log, prodtw
import heapq

import numpy as np


def prime_sieve(n):
    """Numpy boolean array where sieve[i] is True iff i is prime (0 <= i <= n)."""
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    for i in range(2, isqrt(n) + 1):
        if sieve[i]:
            sieve[i * i::i] = False
    return sieve


def primes_upto(n):
    """List of all primes <= n."""
    return np.flatnonzero(prime_sieve(n)).tolist()


def is_prime(n) -> bool:
    """Deterministic Miller-Rabin primality test."""
    if n < 2:
        return False
    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if n % p == 0:
            return n == p
    if n < 41 * 41:
        return True
    bases = (2, 3, 5, 7) if n < 3_215_031_751 else (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    d, r = n - 1, 0
    while d % 2 == 0:
        d, r = d // 2, r + 1
    for a in bases:
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(r - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True


def factorize(n):
    """Prime factorization of n as a dict of {prime: exponent}."""
    f = {}
    while n % 2 == 0:
        f[2] = f.get(2, 0) + 1
        n //= 2
    p = 3
    while p * p <= n:
        while n % p == 0:
            f[p] = f.get(p, 0) + 1
            n //= p
        p += 2
    if n > 1:
        f[n] = f.get(n, 0) + 1
    return f


def num_divisors(n):
    """d(n): the number of divisors of n."""
    return prod(e + 1 for e in factorize(n).values())


def sum_divisors(n):
    """sigma(n): the sum of all divisors of n (including n)."""
    return prod((p ** (e + 1) - 1) // (p - 1) for p, e in factorize(n).items())


def proper_divisor_sum(n):
    """Sum of the proper divisors of n (excluding n itself)."""
    return sum_divisors(n) - n


def divisor_sum_sieve(n):
    """Numpy array s where s[i] = sum of the proper divisors of i (0 <= i <= n)."""
    s = np.zeros(n + 1, dtype=np.int64)
    for d in range(1, isqrt(n) + 1):
        k = np.arange(d, n // d + 1)   # cofactors of d, so that d*k <= n
        s[d * k] += d + k              # add the divisor pair (d, k) to d*k
        s[d * d] -= d                  # d*d got d added twice just above
    return s - np.arange(n + 1)        # make it proper: remove i itself


def digit_sum(n):
    """Sum of the decimal digits of n."""
    return sum(map(int, str(n)))


def is_palindrome(n):
    """Does n (or its string form) read the same forward and backward?"""
    s = str(n)
    return s == s[::-1]


def is_permutation(a, b):
    """Are a and b digit-permutations of each other?"""
    return sorted(str(a)) == sorted(str(b))


def read_matrix(text):
    """Parse a grid of numbers (comma- or space-separated) into a list of rows."""
    return [[int(x) for x in line.replace(',', ' ').split()]
            for line in text.strip().splitlines()]


def max_path(rows):
    """Maximum top-to-bottom path sum in a triangle of numbers."""
    dp = rows[-1][:]
    for row in reversed(rows[:-1]):
        dp = [x + max(a, b) for x, a, b in zip(row, dp, dp[1:])]
    return dp[0]


def triangle(n):  return n * (n + 1) // 2
def pentagon(n):  return n * (3 * n - 1) // 2
def hexagon(n):   return n * (2 * n - 1)


def is_pentagonal(p):
    """Is p a pentagonal number?"""
    r = isqrt(24 * p + 1)
    return r * r == 24 * p + 1 and r % 6 == 5

## [Problem 1](https://projecteuler.net/problem=1)

*Closed-form arithmetic series with inclusion-exclusion: no loop needed.*

In [3]:
def euler_1():
    """Multiples of 3 or 5"""
    def multiples_sum(k, limit=999):
        m = limit // k
        return k * m * (m + 1) // 2
    return multiples_sum(3) + multiples_sum(5) - multiples_sum(15)


run(euler_1)

  1: Multiples of 3 or 5                           0 msec ⇒ 233168            ✅ 

## [Problem 2](https://projecteuler.net/problem=2)

*Iterate the Fibonacci sequence, summing the even terms.*

In [4]:
def euler_2():
    """Even Fibonacci Numbers"""
    total, a, b = 0, 1, 2
    while b <= 4_000_000:
        if b % 2 == 0:
            total += b
        a, b = b, a + b
    return total


run(euler_2)

  2: Even Fibonacci Numbers                        0 msec ⇒ 4613732           ✅ 

## [Problem 3](https://projecteuler.net/problem=3)

*Trial-division factorization; the answer is the largest prime found.*

In [5]:
def euler_3():
    """Largest Prime Factor"""
    return max(factorize(600851475143))


run(euler_3)

  3: Largest Prime Factor                          0 msec ⇒ 6857              ✅ 

## [Problem 4](https://projecteuler.net/problem=4)

*Descending double loop with early breaks once no product can beat the best.*

In [6]:
def euler_4():
    """Largest Palindrome Product"""
    best = 0
    for a in range(999, 99, -1):
        if a * 999 <= best:
            break
        for b in range(999, a - 1, -1):
            p = a * b
            if p <= best:
                break
            if is_palindrome(p):
                best = p
    return best


run(euler_4)

  4: Largest Palindrome Product                    0 msec ⇒ 906609            ✅ 

## [Problem 5](https://projecteuler.net/problem=5)

*The answer is exactly lcm(1..20).*

In [7]:
def euler_5():
    """Smallest Multiple"""
    return lcm(*range(1, 21))


run(euler_5)

  5: Smallest Multiple                             0 msec ⇒ 232792560         ✅ 

## [Problem 6](https://projecteuler.net/problem=6)

*Closed forms for the sum and the sum of squares.*

In [8]:
def euler_6():
    """Sum Square Difference"""
    n = 100
    return (n * (n + 1) // 2) ** 2 - n * (n + 1) * (2 * n + 1) // 6


run(euler_6)

  6: Sum Square Difference                         0 msec ⇒ 25164150          ✅ 

## [Problem 7](https://projecteuler.net/problem=7)

*Sieve a comfortable upper bound and index into the list of primes.*

In [9]:
def euler_7():
    """10001st Prime"""
    return primes_upto(120_000)[10_000]


run(euler_7)

  7: 10001st Prime                                 1 msec ⇒ 104743            ✅ 

## [Problem 8](https://projecteuler.net/problem=8)

*Slide a window of 13 digits across the series, taking the max product.*

In [10]:
def euler_8():
    """Largest Product in a Series"""
    digits = [int(c) for c in ''.join(DATA[8].split())]
    return max(prod(digits[i:i + 13]) for i in range(len(digits) - 12))


run(euler_8)

  8: Largest Product in a Series                   0 msec ⇒ 23514624000       ✅ 

## [Problem 9](https://projecteuler.net/problem=9)

*Loop over a and b; c is determined by the perimeter.*

In [11]:
def euler_9():
    """Special Pythagorean Triplet"""
    for a in range(1, 333):
        for b in range(a + 1, (1000 - a) // 2):
            c = 1000 - a - b
            if a * a + b * b == c * c:
                return a * b * c


run(euler_9)

  9: Special Pythagorean Triplet                   4 msec ⇒ 31875000          ✅ 

## [Problem 10](https://projecteuler.net/problem=10)

*Numpy sieve of Eratosthenes; sum the surviving indexes.*

In [12]:
def euler_10():
    """Summation of Primes"""
    return int(np.flatnonzero(prime_sieve(2_000_000)).sum())


run(euler_10)

 10: Summation of Primes                           4 msec ⇒ 142913828922      ✅ 

## [Problem 11](https://projecteuler.net/problem=11)

*Check the product of 4 in a row in each of the 4 directions from every cell.*

In [13]:
def euler_11():
    """Largest Product in a Grid"""
    g = read_matrix(DATA[11])
    best = 0
    for r in range(20):
        for c in range(20):
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                if 0 <= r + 3 * dr < 20 and 0 <= c + 3 * dc < 20:
                    best = max(best, prod(g[r + i * dr][c + i * dc] for i in range(4)))
    return best


run(euler_11)

 11: Largest Product in a Grid                     0 msec ⇒ 70600674          ✅ 

## [Problem 12](https://projecteuler.net/problem=12)

*triangle(n) = n(n+1)/2 splits into two coprime halves, so d(a*b) = d(a)*d(b).*

In [14]:
def euler_12():
    """Highly Divisible Triangular Number"""
    for n in count(2):
        a, b = (n // 2, n + 1) if n % 2 == 0 else (n, (n + 1) // 2)
        if num_divisors(a) * num_divisors(b) > 500:
            return a * b


run(euler_12)

 12: Highly Divisible Triangular Number           27 msec ⇒ 76576500          ✅ 

## [Problem 13](https://projecteuler.net/problem=13)

*Python big ints make this a one-liner.*

In [15]:
def euler_13():
    """Large Sum"""
    return int(str(sum(int(line) for line in DATA[13].split()))[:10])


run(euler_13)

 13: Large Sum                                     0 msec ⇒ 5537376230        ✅ 

## [Problem 14](https://projecteuler.net/problem=14)

*Chain lengths for even n cost O(1); odd n walk down to a smaller cached value.*

In [16]:
def euler_14():
    """Longest Collatz Sequence"""
    N = 1_000_000
    L = [0] * N
    L[1] = 1
    best_n, best = 1, 1
    for n in range(2, N):
        if n & 1:
            m, steps = (3 * n + 1) >> 1, 2
            while m >= n:                      # walk until below n (already cached)
                if m & 1:
                    m, steps = (3 * m + 1) >> 1, steps + 2
                else:
                    m, steps = m >> 1, steps + 1
            v = L[m] + steps
        else:
            v = L[n >> 1] + 1
        L[n] = v
        if v > best:
            best_n, best = n, v
    return best_n


run(euler_14)

 14: Longest Collatz Sequence                    192 msec ⇒ 837799            ✅ 

## [Problem 15](https://projecteuler.net/problem=15)

*The paths are exactly the ways to arrange 20 rights and 20 downs: C(40, 20).*

In [17]:
def euler_15():
    """Lattice Paths"""
    return comb(40, 20)


run(euler_15)

 15: Lattice Paths                                 0 msec ⇒ 137846528820      ✅ 

## [Problem 16](https://projecteuler.net/problem=16)

*Python big ints again.*

In [18]:
def euler_16():
    """Power Digit Sum"""
    return digit_sum(2 ** 1000)


run(euler_16)

 16: Power Digit Sum                               0 msec ⇒ 1366              ✅ 

## [Problem 17](https://projecteuler.net/problem=17)

*Table of letter counts for the number words; assemble counts positionally.*

In [19]:
def euler_17():
    """Number Letter Counts"""
    ones = [0, 3, 3, 5, 4, 4, 3, 5, 5, 4,          # zero .. nine
            3, 6, 6, 8, 8, 7, 7, 9, 8, 8]          # ten .. nineteen
    tens = [0, 0, 6, 6, 5, 5, 5, 7, 6, 6]          # -, -, twenty .. ninety
    def letters(n):
        if n == 1000:
            return 3 + 8                           # "one" + "thousand"
        h, r = divmod(n, 100)
        total = ones[h] + 7 + (3 if r else 0) if h else 0   # "hundred" (+ "and")
        return total + (ones[r] if r < 20 else tens[r // 10] + ones[r % 10])
    return sum(letters(n) for n in range(1, 1001))


run(euler_17)

 17: Number Letter Counts                          0 msec ⇒ 21124             ✅ 

## [Problem 18](https://projecteuler.net/problem=18)

*Bottom-up dynamic programming over the triangle.*

In [20]:
def euler_18():
    """Maximum Path Sum I"""
    return max_path(read_matrix(DATA[18]))


run(euler_18)

 18: Maximum Path Sum I                            0 msec ⇒ 1074              ✅ 

## [Problem 19](https://projecteuler.net/problem=19)

*Let the datetime module do the calendar arithmetic.*

In [21]:
def euler_19():
    """Counting Sundays"""
    return sum(date(y, m, 1).weekday() == 6
               for y in range(1901, 2001) for m in range(1, 13))


run(euler_19)

 19: Counting Sundays                              0 msec ⇒ 171               ✅ 

## [Problem 20](https://projecteuler.net/problem=20)

*Big ints once more.*

In [22]:
def euler_20():
    """Factorial Digit Sum"""
    return digit_sum(factorial(100))


run(euler_20)

 20: Factorial Digit Sum                           0 msec ⇒ 648               ✅ 

## [Problem 21](https://projecteuler.net/problem=21)

*Proper-divisor sums from factorization; check the amicable condition.*

In [23]:
def euler_21():
    """Amicable Numbers"""
    total = 0
    for n in range(2, 10000):
        m = proper_divisor_sum(n)
        if m != n and proper_divisor_sum(m) == n:
            total += n
    return total


run(euler_21)

 21: Amicable Numbers                             21 msec ⇒ 31626             ✅ 

## [Problem 22](https://projecteuler.net/problem=22)

*Sort the names; score = position times alphabetical value.*

In [24]:
def euler_22():
    """Names Scores"""
    names = sorted(DATA[22].replace('"', '').split(','))
    return sum(i * sum(ord(c) - 64 for c in name)
               for i, name in enumerate(names, 1))


run(euler_22)

 22: Names Scores                                  2 msec ⇒ 871198282         ✅ 

## [Problem 23](https://projecteuler.net/problem=23)

*Big-int bitset: OR together shifted copies to mark all abundant-pair sums at C speed.*

In [25]:
def euler_23():
    """Non-Abundant Sums"""
    LIMIT = 28123
    s = divisor_sum_sieve(LIMIT)
    abundant = [int(n) for n in np.flatnonzero(s > np.arange(LIMIT + 1))]
    mask = 0
    for a in abundant:
        mask |= 1 << a
    sums = 0
    for a in abundant:
        sums |= mask << a          # marks every abundant + a in one big-int OR
    bits = bin(sums)[2:][::-1]     # bits[n] == '1' iff n is a sum of two abundants
    return sum(n for n in range(1, LIMIT + 1)
               if n >= len(bits) or bits[n] == '0')


run(euler_23)

 23: Non-Abundant Sums                            11 msec ⇒ 4179871           ✅ 

## [Problem 24](https://projecteuler.net/problem=24)

*Factorial number system: pick each digit directly, no enumeration.*

In [26]:
def euler_24():
    """Lexicographic Permutations"""
    digits, n, out = list('0123456789'), 999_999, ''
    for k in range(9, -1, -1):
        i, n = divmod(n, factorial(k))
        out += digits.pop(i)
    return int(out)


run(euler_24)

 24: Lexicographic Permutations                    0 msec ⇒ 2783915460        ✅ 

## [Problem 25](https://projecteuler.net/problem=25)

*Iterate Fibonacci until it reaches 10^999.*

In [27]:
def euler_25():
    """1000-digit Fibonacci Number"""
    target = 10 ** 999
    a, b, n = 1, 1, 2
    while b < target:
        a, b, n = b, a + b, n + 1
    return n


run(euler_25)

 25: 1000-digit Fibonacci Number                   0 msec ⇒ 4782              ✅ 

## [Problem 26](https://projecteuler.net/problem=26)

*The cycle length is the multiplicative order of 10 mod d (after removing 2s and 5s).*

In [28]:
def euler_26():
    """Reciprocal Cycles"""
    def cycle_length(d):
        while d % 2 == 0: d //= 2
        while d % 5 == 0: d //= 5
        if d == 1:
            return 0
        k, r = 1, 10 % d
        while r != 1:
            k, r = k + 1, r * 10 % d
        return k
    return max(range(2, 1000), key=cycle_length)


run(euler_26)

 26: Reciprocal Cycles                             4 msec ⇒ 983               ✅ 

## [Problem 27](https://projecteuler.net/problem=27)

*b must be prime; test chains against a precomputed sieve (as bytes, for fast lookup).*

In [29]:
def euler_27():
    """Quadratic Primes"""
    sieve = prime_sieve(2_000_000).tobytes()
    def chain(a, b):
        n = 0
        while (v := n * n + a * n + b) > 1 and sieve[v]:
            n += 1
        return n
    _, a, b = max((chain(a, b), a, b)
                  for b in primes_upto(999) for a in range(-999, 1000, 2))
    return a * b


run(euler_27)

 27: Quadratic Primes                             28 msec ⇒ -59231            ✅ 

## [Problem 28](https://projecteuler.net/problem=28)

*The ring at distance k contributes 4(2k+1)^2 - 12k; sum the closed form.*

In [30]:
def euler_28():
    """Number Spiral Diagonals"""
    return 1 + sum(4 * (2 * k + 1) ** 2 - 12 * k for k in range(1, 501))


run(euler_28)

 28: Number Spiral Diagonals                       0 msec ⇒ 669171001         ✅ 

## [Problem 29](https://projecteuler.net/problem=29)

*A set comprehension over all 99 x 99 powers.*

In [31]:
def euler_29():
    """Distinct Powers"""
    return len({a ** b for a in range(2, 101) for b in range(2, 101)})


run(euler_29)

 29: Distinct Powers                               2 msec ⇒ 9183              ✅ 

## [Problem 30](https://projecteuler.net/problem=30)

*Enumerate digit multisets (order is irrelevant to the sum), not all numbers.*

In [32]:
def euler_30():
    """Digit Fifth Powers"""
    p5 = [d ** 5 for d in range(10)]
    total = 0
    for k in range(2, 8):
        for combo in combinations_with_replacement(range(10), k):
            s = sum(p5[d] for d in combo)
            if sorted(map(int, str(s))) == list(combo):
                total += s
    return total


run(euler_30)

 30: Digit Fifth Powers                           11 msec ⇒ 443839            ✅ 

## [Problem 31](https://projecteuler.net/problem=31)

*Classic coin-counting dynamic program.*

In [33]:
def euler_31():
    """Coin Sums"""
    ways = [1] + [0] * 200
    for c in (1, 2, 5, 10, 20, 50, 100, 200):
        for v in range(c, 201):
            ways[v] += ways[v - c]
    return ways[200]


run(euler_31)

 31: Coin Sums                                     0 msec ⇒ 73682             ✅ 

## [Problem 32](https://projecteuler.net/problem=32)

*Only 1x4 and 2x3 digit splits can make 9 digits total; collect distinct products.*

In [34]:
def euler_32():
    """Pandigital Products"""
    products = set()
    for a in range(2, 100):
        start = 1234 if a < 10 else 123
        for b in range(start, 10000 // a + 1):
            s = f'{a}{b}{a * b}'
            if len(s) == 9 and set(s) == set('123456789'):
                products.add(a * b)
    return sum(products)


run(euler_32)

 32: Pandigital Products                           8 msec ⇒ 45228             ✅ 

## [Problem 33](https://projecteuler.net/problem=33)

*Try cancelling each shared nonzero digit; keep fractions where it works.*

In [35]:
def euler_33():
    """Digit Cancelling Fractions"""
    found = []
    for n in range(10, 100):
        for d in range(n + 1, 100):
            for c in set(str(n)) & set(str(d)) - {'0'}:
                n2 = int(str(n).replace(c, '', 1))
                d2 = int(str(d).replace(c, '', 1))
                if d2 and n * d2 == n2 * d:
                    found.append(Fraction(n, d))
                    break
    return prod(found).denominator


run(euler_33)

 33: Digit Cancelling Fractions                    1 msec ⇒ 100               ✅ 

## [Problem 34](https://projecteuler.net/problem=34)

*Same digit-multiset trick as Problem 30, with factorials.*

In [36]:
def euler_34():
    """Digit Factorials"""
    f = [factorial(d) for d in range(10)]
    total = 0
    for k in range(2, 8):
        for combo in combinations_with_replacement(range(10), k):
            s = sum(f[d] for d in combo)
            if sorted(map(int, str(s))) == list(combo):
                total += s
    return total


run(euler_34)

 34: Digit Factorials                             12 msec ⇒ 40730             ✅ 

## [Problem 35](https://projecteuler.net/problem=35)

*Sieve once; multi-digit candidates may only use digits 1, 3, 7, 9.*

In [37]:
def euler_35():
    """Circular Primes"""
    sieve = prime_sieve(1_000_000).tobytes()
    total = 0
    for p in primes_upto(999_999):
        s = str(p)
        if len(s) > 1 and set(s) & set('024568'):
            continue
        if all(sieve[int(s[i:] + s[:i])] for i in range(1, len(s))):
            total += 1
    return total


run(euler_35)

 35: Circular Primes                              30 msec ⇒ 55                ✅ 

## [Problem 36](https://projecteuler.net/problem=36)

*Generate the ~2000 decimal palindromes below a million; test each in binary.*

In [38]:
def euler_36():
    """Double-base Palindromes"""
    total = 0
    for half in range(1, 1000):
        h = str(half)
        for pal in (h + h[::-1], h + h[::-1][1:]):   # even and odd lengths
            v = int(pal)
            if v < 1_000_000 and bin(v)[2:] == bin(v)[:1:-1]:
                total += v
    return total


run(euler_36)

 36: Double-base Palindromes                       0 msec ⇒ 872187            ✅ 

## [Problem 37](https://projecteuler.net/problem=37)

*Grow right-truncatable primes digit by digit; test each for left-truncatability.*

In [39]:
def euler_37():
    """Truncatable Primes"""
    total, frontier = 0, [2, 3, 5, 7]
    while frontier:
        new = []
        for p in frontier:
            for d in (1, 3, 7, 9):
                q = p * 10 + d
                if is_prime(q):
                    new.append(q)
                    s = str(q)
                    if all(is_prime(int(s[i:])) for i in range(1, len(s))):
                        total += q
        frontier = new
    return total


run(euler_37)

 37: Truncatable Primes                            0 msec ⇒ 748317            ✅ 

## [Problem 38](https://projecteuler.net/problem=38)

*For each multiplier count n, x can have at most 9//n digits.*

In [40]:
def euler_38():
    """Pandigital Multiples"""
    best = 0
    for n in range(2, 10):
        for x in range(1, 10 ** (9 // n)):
            s = ''.join(str(x * i) for i in range(1, n + 1))
            if len(s) == 9 and set(s) == set('123456789'):
                best = max(best, int(s))
    return best


run(euler_38)

 38: Pandigital Multiples                          4 msec ⇒ 932718654         ✅ 

## [Problem 39](https://projecteuler.net/problem=39)

*Generate primitive triples with Euclid's formula; count all multiples per perimeter.*

In [41]:
def euler_39():
    """Integer Right Triangles"""
    counts = Counter()
    for m in range(2, isqrt(500) + 1):
        for n in range(1 + m % 2, m, 2):
            if gcd(m, n) == 1:
                p = 2 * m * (m + n)
                for q in range(p, 1001, p):
                    counts[q] += 1
    return counts.most_common(1)[0][0]


run(euler_39)

 39: Integer Right Triangles                       0 msec ⇒ 840               ✅ 

## [Problem 40](https://projecteuler.net/problem=40)

*Jump straight to the k-th digit by skipping whole blocks of same-length numbers.*

In [42]:
def euler_40():
    """Champernowne's Constant"""
    def digit(k):
        length, first, block = 1, 1, 9
        while k > length * block:
            k -= length * block
            length, first, block = length + 1, first * 10, block * 10
        num = first + (k - 1) // length
        return int(str(num)[(k - 1) % length])
    return prod(digit(10 ** i) for i in range(7))


run(euler_40)

 40: Champernowne's Constant                       0 msec ⇒ 210               ✅ 

## [Problem 41](https://projecteuler.net/problem=41)

*8- and 9-digit pandigitals are divisible by 3, so try 7 digits (then 4) in descending order.*

In [43]:
def euler_41():
    """Pandigital Prime"""
    for k in (7, 4):
        for p in permutations('7654321'[7 - k:]):
            v = int(''.join(p))
            if is_prime(v):
                return v


run(euler_41)

 41: Pandigital Prime                              0 msec ⇒ 7652413           ✅ 

## [Problem 42](https://projecteuler.net/problem=42)

*Score each word; check membership in a set of triangle numbers.*

In [44]:
def euler_42():
    """Coded Triangle Numbers"""
    words = DATA[42].replace('"', '').split(',')
    triangles = {triangle(n) for n in range(1, 30)}
    return sum(sum(ord(c) - 64 for c in w) in triangles for w in words)


run(euler_42)

 42: Coded Triangle Numbers                        1 msec ⇒ 162               ✅ 

## [Problem 43](https://projecteuler.net/problem=43)

*Build the number right to left, keeping only prefixes that satisfy each divisibility test.*

In [45]:
def euler_43():
    """Sub-string Divisibility"""
    strings = [f'{m:03d}' for m in range(0, 1000, 17) if len(set(f'{m:03d}')) == 3]
    for p in (13, 11, 7, 5, 3, 2):
        strings = [d + s for s in strings for d in '0123456789'
                   if d not in s and int(d + s[:2]) % p == 0]
    return sum(int((set('0123456789') - set(s)).pop() + s) for s in strings)


run(euler_43)

 43: Sub-string Divisibility                       0 msec ⇒ 16695334890       ✅ 

## [Problem 44](https://projecteuler.net/problem=44)

*Scan pairs in order of difference-candidate k, breaking once no pair can improve.*

In [46]:
def euler_44():
    """Pentagon Numbers"""
    pentagons = [pentagon(n) for n in range(1, 2500)]
    pset = set(pentagons)
    best = None
    for k in range(1, len(pentagons)):
        pk = pentagons[k]
        for j in range(k - 1, -1, -1):
            d = pk - pentagons[j]
            if best and d >= best:
                break
            if d in pset and pk + pentagons[j] in pset:
                best = d
    return best


run(euler_44)

 44: Pentagon Numbers                             86 msec ⇒ 5482660           ✅ 

## [Problem 45](https://projecteuler.net/problem=45)

*Every hexagonal number is triangular, so just scan hexagonals for pentagonality.*

In [47]:
def euler_45():
    """Triangular, Pentagonal, and Hexagonal"""
    for n in count(144):
        h = hexagon(n)
        if is_pentagonal(h):
            return h


run(euler_45)

 45: Triangular, Pentagonal, and Hexagonal         4 msec ⇒ 1533776805        ✅ 

## [Problem 46](https://projecteuler.net/problem=46)

*For each odd composite, try subtracting twice a square and testing for primality.*

In [48]:
def euler_46():
    """Goldbach's Other Conjecture"""
    for n in count(9, 2):
        if not is_prime(n):
            if not any(is_prime(n - 2 * k * k)
                       for k in range(1, isqrt(n // 2) + 1)):
                return n


run(euler_46)

 46: Goldbach's Other Conjecture                   6 msec ⇒ 5777              ✅ 

## [Problem 47](https://projecteuler.net/problem=47)

*Sieve the count of distinct prime factors; find 4 consecutive 4s with numpy.*

In [49]:
def euler_47():
    """Distinct Primes Factors"""
    N = 200_000
    counts = np.zeros(N, dtype=np.int8)
    for p in range(2, N):
        if counts[p] == 0:
            counts[p::p] += 1
    ok = counts == 4
    hits = ok[:-3] & ok[1:-2] & ok[2:-1] & ok[3:]
    return int(np.flatnonzero(hits)[0])


run(euler_47)

 47: Distinct Primes Factors                      24 msec ⇒ 134043            ✅ 

## [Problem 48](https://projecteuler.net/problem=48)

*Modular exponentiation keeps every term to 10 digits.*

In [50]:
def euler_48():
    """Self Powers"""
    M = 10 ** 10
    return sum(pow(n, n, M) for n in range(1, 1001)) % M


run(euler_48)

 48: Self Powers                                   1 msec ⇒ 9110846700        ✅ 

## [Problem 49](https://projecteuler.net/problem=49)

*Group 4-digit primes by digit signature; look for an arithmetic triple in each group.*

In [51]:
def euler_49():
    """Prime Permutations"""
    groups = defaultdict(list)
    for p in primes_upto(9999):
        if p >= 1000:
            groups[''.join(sorted(str(p)))].append(p)
    for group in groups.values():
        gset = set(group)
        for a, b in combinations(group, 2):
            if 2 * b - a in gset and a != 1487:
                return int(f'{a}{b}{2 * b - a}')


run(euler_49)

 49: Prime Permutations                            1 msec ⇒ 296962999629      ✅ 

## [Problem 50](https://projecteuler.net/problem=50)

*Prefix sums of primes; try window lengths from longest possible downward.*

In [52]:
def euler_50():
    """Consecutive Prime Sum"""
    N = 1_000_000
    ps = primes_upto(N)
    pset = set(ps)
    prefix = [0]
    for p in ps:
        prefix.append(prefix[-1] + p)
    max_len = next(l for l in range(len(prefix) - 1, 0, -1) if prefix[l] < N)
    for length in range(max_len, 0, -1):
        for i in range(len(ps) - length + 1):
            s = prefix[i + length] - prefix[i]
            if s >= N:
                break
            if s in pset:
                return s


run(euler_50)

 50: Consecutive Prime Sum                         7 msec ⇒ 997651            ✅ 

## [Problem 51](https://projecteuler.net/problem=51)

*An 8-family must replace a digit occurring 3 (or 6) times, and that digit must be 0, 1, or 2.*

In [53]:
def euler_51():
    """Prime Digit Replacements"""
    sieve = prime_sieve(1_000_000).tobytes()
    for p in primes_upto(999_999):
        s = str(p)
        for d in '012':
            if s.count(d) in (3, 6):
                family = [int(s.replace(d, r)) for r in '0123456789']
                members = [q for q in family if len(str(q)) == len(s) and sieve[q]]
                if len(members) == 8:
                    return members[0]


run(euler_51)

 51: Prime Digit Replacements                      6 msec ⇒ 121313            ✅ 

## [Problem 52](https://projecteuler.net/problem=52)

*x and 6x must have the same digit count, so only scan [10^k, 10^(k+1)/6).*

In [54]:
def euler_52():
    """Permuted Multiples"""
    for k in count(5):
        for n in range(10 ** k, 10 ** (k + 1) // 6 + 1):
            if all(is_permutation(n, m * n) for m in range(2, 7)):
                return n


run(euler_52)

 52: Permuted Multiples                           20 msec ⇒ 142857            ✅ 

## [Problem 53](https://projecteuler.net/problem=53)

*math.comb is exact; just count.*

In [55]:
def euler_53():
    """Combinatoric Selections"""
    return sum(comb(n, r) > 1_000_000
               for n in range(1, 101) for r in range(n + 1))


run(euler_53)

 53: Combinatoric Selections                       0 msec ⇒ 4075              ✅ 

## [Problem 54](https://projecteuler.net/problem=54)

*Rank each hand as (category, tiebreak values); compare the tuples.*

In [56]:
def euler_54():
    """Poker Hands"""
    def rank(hand):
        values = sorted(('23456789TJQKA'.index(c[0]) for c in hand), reverse=True)
        groups = Counter(values)
        by_count = sorted(groups.items(), key=lambda kv: (kv[1], kv[0]), reverse=True)
        counts = tuple(c for v, c in by_count)
        order = tuple(v for v, c in by_count)
        flush = len({c[1] for c in hand}) == 1
        straight = counts == (1, 1, 1, 1, 1) and values[0] - values[4] == 4
        category = (8 if straight and flush else
                    7 if counts == (4, 1) else
                    6 if counts == (3, 2) else
                    5 if flush else
                    4 if straight else
                    3 if counts == (3, 1, 1) else
                    2 if counts == (2, 2, 1) else
                    1 if counts == (2, 1, 1, 1) else 0)
        return (category, order)
    return sum(rank(cards[:5]) > rank(cards[5:])
               for line in DATA[54].splitlines()
               for cards in [line.split()])


run(euler_54)

 54: Poker Hands                                   4 msec ⇒ 376               ✅ 

## [Problem 55](https://projecteuler.net/problem=55)

*Direct simulation: at most 50 reverse-and-add steps for each of 10000 numbers.*

In [57]:
def euler_55():
    """Lychrel Numbers"""
    def is_lychrel(n):
        for _ in range(50):
            n += int(str(n)[::-1])
            if is_palindrome(n):
                return False
        return True
    return sum(is_lychrel(n) for n in range(1, 10000))


run(euler_55)

 55: Lychrel Numbers                               9 msec ⇒ 249               ✅ 

## [Problem 56](https://projecteuler.net/problem=56)

*Brute force over all 99 x 99 powers; big ints keep it exact.*

In [58]:
def euler_56():
    """Powerful Digit Sum"""
    return max(digit_sum(a ** b) for a in range(1, 100) for b in range(1, 100))


run(euler_56)

 56: Powerful Digit Sum                           23 msec ⇒ 972               ✅ 

## [Problem 57](https://projecteuler.net/problem=57)

*Iterate the convergent recurrence; compare digit counts.*

In [59]:
def euler_57():
    """Square Root Convergents"""
    n, d, total = 3, 2, 0
    for _ in range(1000):
        if len(str(n)) > len(str(d)):
            total += 1
        n, d = n + 2 * d, n + d
    return total


run(euler_57)

 57: Square Root Convergents                       1 msec ⇒ 153               ✅ 

## [Problem 58](https://projecteuler.net/problem=58)

*Only 3 of the 4 corners can be prime (the 4th is a square); Miller-Rabin each corner.*

In [60]:
def euler_58():
    """Spiral Primes"""
    nprimes, ncorners, side = 0, 1, 1
    while True:
        side += 2
        sq = side * side
        nprimes += sum(is_prime(sq - k * (side - 1)) for k in (1, 2, 3))
        ncorners += 4
        if nprimes * 10 < ncorners:
            return side


run(euler_58)

 58: Spiral Primes                                35 msec ⇒ 26241             ✅ 

## [Problem 59](https://projecteuler.net/problem=59)

*Each key byte acts on every 3rd character independently: pick the one making the most spaces.*

In [61]:
def euler_59():
    """XOR Decryption"""
    codes = [int(x) for x in DATA[59].split(',')]
    key = [max(range(97, 123), key=lambda k: sum(c ^ k == 32 for c in codes[off::3]))
           for off in range(3)]
    return sum(c ^ key[i % 3] for i, c in enumerate(codes))


run(euler_59)

 59: XOR Decryption                                1 msec ⇒ 107359            ✅ 

## [Problem 60](https://projecteuler.net/problem=60)

*Primes pair only within their residue class mod 3; DFS with sum-based pruning.*

In [62]:
def euler_60():
    """Prime Pair Sets"""
    candidates = [p for p in primes_upto(9000) if p not in (2, 5)]
    @lru_cache(maxsize=None)
    def pair(p, q):
        return (is_prime(int(f'{p}{q}')) and is_prime(int(f'{q}{p}')))
    best = inf
    def extend(chain, total, cands):
        nonlocal best
        if len(chain) == 5:
            best = min(best, total)
            return
        need = 5 - len(chain)
        for i, p in enumerate(cands):
            if total + p * need >= best:
                break
            extend(chain + [p], total + p,
                   [q for q in cands[i + 1:] if pair(p, q)])
    for cls in (1, 2):
        group = [p for p in candidates if p % 3 == cls or p == 3]
        extend([], 0, group)
    return best


run(euler_60)

 60: Prime Pair Sets                             472 msec ⇒ 26033             ✅ 

## [Problem 61](https://projecteuler.net/problem=61)

*DFS over which figurate type comes next, linking on the 2-digit overlap.*

In [63]:
def euler_61():
    """Cyclical Figurate Numbers"""
    def figurates(k):
        out, n = [], 1
        while (v := n * ((k - 2) * n - (k - 4)) // 2) < 10000:
            if v >= 1000:
                out.append(v)
            n += 1
        return out
    sets = {k: figurates(k) for k in range(3, 9)}
    def dfs(chain, remaining):
        if not remaining:
            if chain[-1] % 100 == chain[0] // 100:
                return sum(chain)
            return None
        for k in remaining:
            for v in sets[k]:
                if v // 100 == chain[-1] % 100:
                    if (answer := dfs(chain + [v], remaining - {k})) is not None:
                        return answer
        return None
    for start in sets[8]:
        if (answer := dfs([start], frozenset({3, 4, 5, 6, 7}))) is not None:
            return answer


run(euler_61)

 61: Cyclical Figurate Numbers                     0 msec ⇒ 28684             ✅ 

## [Problem 62](https://projecteuler.net/problem=62)

*Group cubes by digit signature; stop at the first signature with 5 cubes.*

In [64]:
def euler_62():
    """Cubic Permutations"""
    groups = defaultdict(list)
    for n in count(1):
        c = n ** 3
        sig = ''.join(sorted(str(c)))
        groups[sig].append(c)
        if len(groups[sig]) == 5:
            return groups[sig][0]


run(euler_62)

 62: Cubic Permutations                            4 msec ⇒ 127035954683      ✅ 

## [Problem 63](https://projecteuler.net/problem=63)

*Only bases 1-9 can work (10^n has n+1 digits); count directly.*

In [65]:
def euler_63():
    """Powerful Digit Counts"""
    return sum(len(str(a ** n)) == n
               for a in range(1, 10) for n in range(1, 100))


run(euler_63)

 63: Powerful Digit Counts                         0 msec ⇒ 49                ✅ 

## [Problem 64](https://projecteuler.net/problem=64)

*Standard continued-fraction expansion; the period ends when a = 2*a0.*

In [66]:
def euler_64():
    """Odd Period Square Roots"""
    def period(n):
        a0 = isqrt(n)
        if a0 * a0 == n:
            return 0
        m, d, a, length = 0, 1, a0, 0
        while a != 2 * a0:
            m = d * a - m
            d = (n - m * m) // d
            a = (a0 + m) // d
            length += 1
        return length
    return sum(period(n) % 2 == 1 for n in range(2, 10001))


run(euler_64)

 64: Odd Period Square Roots                      19 msec ⇒ 1322              ✅ 

## [Problem 65](https://projecteuler.net/problem=65)

*e = [2; 1, 2, 1, 1, 4, 1, ...]; run the convergent numerator recurrence.*

In [67]:
def euler_65():
    """Convergents of e"""
    p_prev, p = 1, 2
    for k in range(2, 101):
        a = 2 * k // 3 if k % 3 == 0 else 1
        p_prev, p = p, a * p + p_prev
    return digit_sum(p)


run(euler_65)

 65: Convergents of e                              0 msec ⇒ 272               ✅ 

## [Problem 66](https://projecteuler.net/problem=66)

*Solve each Pell equation exactly with continued-fraction convergents.*

In [68]:
def euler_66():
    """Diophantine Equation"""
    def pell_x(D):
        a0 = isqrt(D)
        m, d, a = 0, 1, a0
        p_prev, p, q_prev, q = 1, a0, 0, 1
        while p * p - D * q * q != 1:
            m = d * a - m
            d = (D - m * m) // d
            a = (a0 + m) // d
            p_prev, p = p, a * p + p_prev
            q_prev, q = q, a * q + q_prev
        return p
    return max((D for D in range(2, 1001) if isqrt(D) ** 2 != D), key=pell_x)


run(euler_66)

 66: Diophantine Equation                          2 msec ⇒ 661               ✅ 

## [Problem 67](https://projecteuler.net/problem=67)

*The same dynamic program as Problem 18 handles 100 rows instantly.*

In [69]:
def euler_67():
    """Maximum Path Sum II"""
    return max_path(read_matrix(DATA[67]))


run(euler_67)

 67: Maximum Path Sum II                           0 msec ⇒ 7273              ✅ 

## [Problem 68](https://projecteuler.net/problem=68)

*Choose the inner ring; the line sum then determines the outer ring completely.*

In [70]:
def euler_68():
    """Magic 5-gon Ring"""
    best = 0
    everything = set(range(1, 11))
    for inner in permutations(range(1, 11), 5):
        if (55 + sum(inner)) % 5:
            continue
        line = (55 + sum(inner)) // 5
        outer = [line - inner[i] - inner[(i + 1) % 5] for i in range(5)]
        if sorted(outer) == sorted(everything - set(inner)):
            i0 = min(range(5), key=lambda i: outer[i])
            s = ''.join(f'{outer[i]}{inner[i]}{inner[(i + 1) % 5]}'
                        for i in ((i0 + j) % 5 for j in range(5)))
            if len(s) == 16:
                best = max(best, int(s))
    return best


run(euler_68)

 68: Magic 5-gon Ring                              6 msec ⇒ 6531031914842725  ✅ 

## [Problem 69](https://projecteuler.net/problem=69)

*n/phi(n) is maximized by the product of the smallest primes.*

In [71]:
def euler_69():
    """Totient Maximum"""
    n = 1
    for p in primes_upto(20):
        if n * p > 1_000_000:
            return n
        n *= p


run(euler_69)

 69: Totient Maximum                               0 msec ⇒ 510510            ✅ 

## [Problem 70](https://projecteuler.net/problem=70)

*The best n is a product of two primes near sqrt(10^7); search that window.*

In [72]:
def euler_70():
    """Totient Permutation"""
    N = 10 ** 7
    ps = [p for p in primes_upto(5000) if p > 1000]
    best, best_ratio = 0, inf
    for i, p in enumerate(ps):
        for q in ps[i + 1:]:
            n = p * q
            if n >= N:
                break
            phi = (p - 1) * (q - 1)
            if n / phi < best_ratio and is_permutation(n, phi):
                best, best_ratio = n, n / phi
    return best


run(euler_70)

 70: Totient Permutation                          25 msec ⇒ 8319823           ✅ 

## [Problem 71](https://projecteuler.net/problem=71)

*The neighbor of 3/7 has 3q = 1 (mod 7); take the largest such denominator.*

In [73]:
def euler_71():
    """Ordered Fractions"""
    q = 1_000_000
    while (3 * q) % 7 != 1:
        q -= 1
    return (3 * q - 1) // 7


run(euler_71)

 71: Ordered Fractions                             0 msec ⇒ 428570            ✅ 

## [Problem 72](https://projecteuler.net/problem=72)

*The totient summatory function via its O(n^(3/4)) divide-and-conquer recurrence.*

In [74]:
def euler_72():
    """Counting Fractions"""
    cache = {}
    def Phi(n):
        """Sum of phi(k) for k = 1..n."""
        if n == 1:
            return 1
        if n in cache:
            return cache[n]
        total = n * (n + 1) // 2
        d = 2
        while d <= n:
            q = n // d
            d_next = n // q + 1
            total -= (d_next - d) * Phi(q)
            d = d_next
        cache[n] = total
        return total
    return Phi(1_000_000) - 1


run(euler_72)

 72: Counting Fractions                           16 msec ⇒ 303963552391      ✅ 

## [Problem 73](https://projecteuler.net/problem=73)

*Count all fractions in range per denominator, then sieve away unreduced ones.*

In [75]:
def euler_73():
    """Counting Fractions in a Range"""
    N = 12000
    F = [(d - 1) // 2 - d // 3 for d in range(N + 1)]
    for d in range(1, N + 1):
        for m in range(2 * d, N + 1, d):
            F[m] -= F[d]
    return sum(F[2:])


run(euler_73)

 73: Counting Fractions in a Range                 5 msec ⇒ 7295372           ✅ 

## [Problem 74](https://projecteuler.net/problem=74)

*Numbers with the same digit multiset have the same chain; count arrangements.*

In [76]:
def euler_74():
    """Digit Factorial Chains"""
    FACT = [factorial(d) for d in range(10)]
    def fsum(n):
        s = 0
        while n:
            s += FACT[n % 10]
            n //= 10
        return s
    L = {}
    def chain_length(n0):
        """Number of distinct terms in the chain starting at n0 (memoized)."""
        path, pos = [], {}
        n = n0
        while n not in L and n not in pos:
            pos[n] = len(path)
            path.append(n)
            n = fsum(n)
        if n in L:
            base, tail = L[n], path
        else:                       # found a new cycle within path
            i = pos[n]
            base, tail = len(path) - i, path[:i]
            for m in path[i:]:
                L[m] = base
        for j, m in enumerate(reversed(tail), 1):
            L[m] = base + j
        return L[n0]
    total = 0
    for k in range(1, 7):
        for combo in combinations_with_replacement(range(10), k):
            if max(combo) == 0:
                continue
            rep = int(''.join(map(str, sorted(combo, reverse=True))))
            if 1 + chain_length(fsum(rep)) == 60:
                counts = Counter(combo)
                arrangements = factorial(k)
                for c in counts.values():
                    arrangements //= factorial(c)
                if counts[0]:       # no leading zero
                    arrangements = arrangements * (k - counts[0]) // k
                total += arrangements
    return total


run(euler_74)

 74: Digit Factorial Chains                        9 msec ⇒ 402               ✅ 

## [Problem 75](https://projecteuler.net/problem=75)

*Euclid's formula for primitive triples; mark all multiples of each perimeter.*

In [77]:
def euler_75():
    """Singular Integer Right Triangles"""
    L = 1_500_000
    times_hit = bytearray(L + 1)
    for m in range(2, isqrt(L // 2) + 1):
        for n in range(1 + m % 2, m, 2):
            if gcd(m, n) == 1:
                p = 2 * m * (m + n)
                if p > L:
                    break
                for q in range(p, L + 1, p):
                    times_hit[q] += 1
    return sum(v == 1 for v in times_hit)


run(euler_75)

 75: Singular Integer Right Triangles             94 msec ⇒ 161667            ✅ 

## [Problem 76](https://projecteuler.net/problem=76)

*Partition-counting dynamic program with parts 1..99.*

In [78]:
def euler_76():
    """Counting Summations"""
    ways = [1] + [0] * 100
    for part in range(1, 100):
        for v in range(part, 101):
            ways[v] += ways[v - part]
    return ways[100]


run(euler_76)

 76: Counting Summations                           0 msec ⇒ 190569291         ✅ 

## [Problem 77](https://projecteuler.net/problem=77)

*The same dynamic program, restricted to prime parts.*

In [79]:
def euler_77():
    """Prime Summations"""
    ways = [1] + [0] * 100
    for p in primes_upto(100):
        for v in range(p, 101):
            ways[v] += ways[v - p]
    return next(n for n in range(2, 101) if ways[n] > 5000)


run(euler_77)

 77: Prime Summations                              0 msec ⇒ 71                ✅ 

## [Problem 78](https://projecteuler.net/problem=78)

*Euler's pentagonal-number recurrence for p(n), mod 10^6, vectorized with numpy.*

In [80]:
def euler_78():
    """Coin Partitions"""
    N = 100_000
    pents, signs = [], []
    j = 1
    while j * (3 * j - 1) // 2 <= N:
        sign = 1 if j % 2 == 1 else -1
        for g in (j * (3 * j - 1) // 2, j * (3 * j + 1) // 2):
            if g <= N:
                pents.append(g)
                signs.append(sign)
        j += 1
    pents = np.array(pents)
    signs = np.array(signs, dtype=np.int64)
    p = np.zeros(N + 1, dtype=np.int64)
    p[0] = 1
    m = 0
    for n in range(1, N + 1):
        while m < len(pents) and pents[m] <= n:
            m += 1
        v = int((signs[:m] * p[n - pents[:m]]).sum()) % 1_000_000
        p[n] = v
        if v == 0:
            return n


run(euler_78)

 78: Coin Partitions                             119 msec ⇒ 55374             ✅ 

## [Problem 79](https://projecteuler.net/problem=79)

*Each digit's set of successors gives a total order: sort by out-degree.*

In [81]:
def euler_79():
    """Passcode Derivation"""
    keys = DATA[79].split()
    after = defaultdict(set)
    digits = set()
    for k in keys:
        digits.update(k)
        after[k[0]].update(k[1:])
        after[k[1]].add(k[2])
    order = sorted(digits, key=lambda d: len(after[d]), reverse=True)
    return int(''.join(order))


run(euler_79)

 79: Passcode Derivation                           0 msec ⇒ 73162890          ✅ 

## [Problem 80](https://projecteuler.net/problem=80)

*isqrt of n * 10^198 yields the first 100 digits exactly, no floating point.*

In [82]:
def euler_80():
    """Square Root Digital Expansion"""
    total = 0
    for n in range(1, 101):
        if isqrt(n) ** 2 != n:
            total += digit_sum(isqrt(n * 10 ** 198))
    return total


run(euler_80)

 80: Square Root Digital Expansion                 0 msec ⇒ 40886             ✅ 

## [Problem 81](https://projecteuler.net/problem=81)

*Row-by-row dynamic program.*

In [83]:
def euler_81():
    """Path Sum: Two Ways"""
    m = read_matrix(DATA[81])
    dp = list(m[0])
    for j in range(1, len(dp)):
        dp[j] += dp[j - 1]
    for row in m[1:]:
        dp[0] += row[0]
        for j in range(1, len(row)):
            dp[j] = row[j] + min(dp[j], dp[j - 1])
    return dp[-1]


run(euler_81)

 81: Path Sum                                      1 msec ⇒ 427337            ✅ 

## [Problem 82](https://projecteuler.net/problem=82)

*Column-by-column dynamic program with an up-pass and a down-pass.*

In [84]:
def euler_82():
    """Path Sum: Three Ways"""
    m = read_matrix(DATA[82])
    n = len(m)
    col = [m[i][0] for i in range(n)]
    for j in range(1, n):
        new = [0] * n
        new[0] = col[0] + m[0][j]
        for i in range(1, n):
            new[i] = m[i][j] + min(col[i], new[i - 1])
        for i in range(n - 2, -1, -1):
            new[i] = min(new[i], new[i + 1] + m[i][j])
        col = new
    return min(col)


run(euler_82)

 82: Path Sum                                      1 msec ⇒ 260324            ✅ 

## [Problem 83](https://projecteuler.net/problem=83)

*Dijkstra's algorithm over the 80x80 grid.*

In [85]:
def euler_83():
    """Path Sum: Four Ways"""
    m = read_matrix(DATA[83])
    n = len(m)
    dist = [[inf] * n for _ in range(n)]
    dist[0][0] = m[0][0]
    frontier = [(m[0][0], 0, 0)]
    while frontier:
        d, r, c = heapq.heappop(frontier)
        if (r, c) == (n - 1, n - 1):
            return d
        if d > dist[r][c]:
            continue
        for r2, c2 in ((r-1, c), (r+1, c), (r, c-1), (r, c+1)):
            if 0 <= r2 < n and 0 <= c2 < n:
                d2 = d + m[r2][c2]
                if d2 < dist[r2][c2]:
                    dist[r2][c2] = d2
                    heapq.heappush(frontier, (d2, r2, c2))


run(euler_83)

 83: Path Sum                                      5 msec ⇒ 425185            ✅ 

## [Problem 84](https://projecteuler.net/problem=84)

*Exact 120-state Markov chain (square x consecutive doubles); no simulation.*

In [86]:
def euler_84():
    """Monopoly Odds"""
    CH, CC, RAILS, UTILS = {7, 22, 36}, {2, 17, 33}, [5, 15, 25, 35], [12, 28]
    def resolve(sq):
        """Distribution over final squares after landing on sq (cards, G2J)."""
        if sq == 30:
            return {10: 1.0}
        out = defaultdict(float)
        if sq in CC:
            out[0] += 1/16; out[10] += 1/16; out[sq] += 14/16
        elif sq in CH:
            next_rail = next(r for r in RAILS + [45] if r > sq) % 40
            next_util = next(u for u in UTILS + [52] if u > sq) % 40
            for target in (0, 10, 11, 24, 39, 5):
                out[target] += 1/16
            out[next_rail] += 2/16
            out[next_util] += 1/16
            for target, p in resolve((sq - 3) % 40).items():
                out[target] += p / 16
            out[sq] += 6/16
        else:
            out[sq] = 1.0
        return out
    T = np.zeros((120, 120))
    for sq in range(40):
        for dbl in range(3):
            state = sq * 3 + dbl
            for i in range(1, 5):
                for j in range(1, 5):
                    if i == j and dbl == 2:
                        T[state, 10 * 3] += 1/16      # third double: go to jail
                        continue
                    dbl2 = dbl + 1 if i == j else 0
                    for target, p in resolve((sq + i + j) % 40).items():
                        T[state, target * 3 + (0 if target == 10 else dbl2)] += p / 16
    v = np.ones(120) / 120
    for _ in range(200):
        v = v @ T
    popularity = v.reshape(40, 3).sum(axis=1)
    top3 = popularity.argsort()[::-1][:3]
    return int(''.join(f'{sq:02d}' for sq in top3))


run(euler_84)

 84: Monopoly Odds                                 2 msec ⇒ 101524            ✅ 

## [Problem 85](https://projecteuler.net/problem=85)

*Grid a x b contains T(a)×T(b) rectangles; scan for the closest to 2 million.*

In [87]:
def euler_85():
    """Counting Rectangles"""
    TARGET = 2_000_000
    best = (inf, 0)
    for a in count(1):
        if triangle(a) ** 2 > 2 * TARGET:
            break
        for b in count(a):
            r = triangle(a) * triangle(b)
            best = min(best, (abs(r - TARGET), a * b))
            if r > TARGET:
                break
    return best[1]


run(euler_85)

 85: Counting Rectangles                           1 msec ⇒ 2772              ✅ 

## [Problem 86](https://projecteuler.net/problem=86)

*For each cuboid dimension M, count integer shortest paths with one numpy sweep.*

In [88]:
def euler_86():
    """Cuboid Route"""
    total, M = 0, 0
    while total < 1_000_000:
        M += 1
        ab = np.arange(2, 2 * M + 1)              # a + b, where 1 <= a <= b <= M
        d2 = ab * ab + M * M
        root = np.rint(np.sqrt(d2)).astype(np.int64)
        ok = root * root == d2
        pairs = ab // 2 - np.maximum(1, ab - M) + 1
        total += int(pairs[ok].sum())
    return M


run(euler_86)

 86: Cuboid Route                                 27 msec ⇒ 1818              ✅ 

## [Problem 87](https://projecteuler.net/problem=87)

*Mark every square+cube+fourth sum in one 50-million-element boolean array.*

In [89]:
def euler_87():
    """Prime Power Triples"""
    N = 50_000_000
    ps = primes_upto(isqrt(N))
    squares = np.array([p * p for p in ps])
    cubes = [p ** 3 for p in ps if p ** 3 < N]
    fourths = [p ** 4 for p in ps if p ** 4 < N]
    seen = np.zeros(N, dtype=bool)
    for f in fourths:
        for c in cubes:
            if c + f >= N:
                break
            s = squares + (c + f)
            seen[s[s < N]] = True
    return int(seen.sum())


run(euler_87)

 87: Prime Power Triples                          22 msec ⇒ 1097343           ✅ 

## [Problem 88](https://projecteuler.net/problem=88)

*DFS over nondecreasing factor lists; k is determined by padding with 1s.*

In [90]:
def euler_88():
    """Product-sum Numbers"""
    K = 12000
    best = [2 * K] * (K + 1)
    def dfs(product, total, nfactors, start):
        k = product - total + nfactors
        if k > K:
            return
        if nfactors >= 2 and product < best[k]:
            best[k] = product
        f = start
        while product * f <= 2 * K:
            dfs(product * f, total + f, nfactors + 1, f)
            f += 1
    dfs(1, 0, 0, 2)
    return sum(set(best[2:]))


run(euler_88)

 88: Product-sum Numbers                          50 msec ⇒ 7587457           ✅ 

## [Problem 89](https://projecteuler.net/problem=89)

*Only 6 patterns are ever written suboptimally; count characters saved by replacement.*

In [91]:
def euler_89():
    """Roman Numerals"""
    saved = 0
    for numeral in DATA[89].split():
        s = numeral
        for old, new in (('DCCCC', 'CM'), ('CCCC', 'CD'), ('LXXXX', 'XC'),
                         ('XXXX', 'XL'), ('VIIII', 'IX'), ('IIII', 'IV')):
            s = s.replace(old, new)
        saved += len(numeral) - len(s)
    return saved


run(euler_89)

 89: Roman Numerals                                0 msec ⇒ 743               ✅ 

## [Problem 90](https://projecteuler.net/problem=90)

*210 possible cubes; test every pair against the 9 squares (with 6 and 9 interchangeable).*

In [92]:
def euler_90():
    """Cube Digit Pairs"""
    SQUARES = [(0, 1), (0, 4), (0, 9), (1, 6), (2, 5), (3, 6), (4, 9), (6, 4), (8, 1)]
    dice = [set(c) | ({6, 9} if {6, 9} & set(c) else set())
            for c in combinations(range(10), 6)]
    return sum(all((a in d1 and b in d2) or (a in d2 and b in d1)
                   for a, b in SQUARES)
               for d1, d2 in combinations(dice, 2))


run(euler_90)

 90: Cube Digit Pairs                              5 msec ⇒ 1217              ✅ 

## [Problem 91](https://projecteuler.net/problem=91)

*3N^2 axis-aligned cases; for the rest, walk perpendicular steps from each point P.*

In [93]:
def euler_91():
    """Right Triangles with Integer Coordinates"""
    N = 50
    total = 3 * N * N
    for x in range(1, N + 1):
        for y in range(1, N + 1):
            g = gcd(x, y)
            dx, dy = x // g, y // g 
            total += min((N - x) * g // y, y * g // x)   # step (dy, -dx)
            total += min(x * g // y, (N - y) * g // x)   # step (-dy, dx)
    return total


run(euler_91)

 91: Right Triangles with Integer Coordinates      0 msec ⇒ 14234             ✅ 

## [Problem 92](https://projecteuler.net/problem=92)

*Numbers with the same digit multiset behave identically; count multisets, not numbers.*

In [94]:
def euler_92():
    """Square Digit Chains"""
    def arrives_at_89(n):
        while n not in (1, 89):
            n = sum(int(d) ** 2 for d in str(n))
        return n == 89
    total = 0
    for combo in combinations_with_replacement(range(10), 7):
        s = sum(d * d for d in combo)
        if s and arrives_at_89(s):
            arrangements = factorial(7)
            for c in Counter(combo).values():
                arrangements //= factorial(c)
            total += arrangements
    return total


run(euler_92)

 92: Square Digit Chains                          25 msec ⇒ 8581146           ✅ 

## [Problem 93](https://projecteuler.net/problem=93)

*Combine value-sets of digit subsets bottom-up, memoized across all digit choices.*

In [95]:
def euler_93():
    """Arithmetic Expressions"""
    memo = {}
    def values(digits):
        """All values reachable using every digit in the frozenset exactly once."""
        if digits in memo:
            return memo[digits]
        if len(digits) == 1:
            result = {float(next(iter(digits)))}
        else:
            result = set()
            smallest = min(digits)
            members = list(digits)
            for r in range(1, len(members)):
                for left in combinations(members, r):
                    if smallest not in left:
                        continue
                    A, B = frozenset(left), digits - frozenset(left)
                    for a in values(A):
                        for b in values(B):
                            result.update((a + b, a - b, b - a, a * b))
                            if abs(b) > 1e-9:
                                result.add(a / b)
                            if abs(a) > 1e-9:
                                result.add(b / a)
        memo[digits] = result
        return result
    best, best_digits = 0, None
    for c in combinations(range(1, 10), 4):
        reachable = {round(v) for v in values(frozenset(c))
                     if v > 0.5 and abs(v - round(v)) < 1e-6}
        n = 1
        while n in reachable:
            n += 1
        if n - 1 > best:
            best, best_digits = n - 1, c
    return int(''.join(map(str, best_digits)))


run(euler_93)

 93: Arithmetic Expressions                       23 msec ⇒ 1258              ✅ 

## [Problem 94](https://projecteuler.net/problem=94)

*Both families come from Pell solutions of u^2 - 3v^2 = 1; no search at all.*

In [96]:
def euler_94():
    """Almost Equilateral Triangles"""
    LIMIT = 10 ** 9
    total = 0
    u, v = 2, 1
    while True:
        pA = 3 * (u * u + v * v) + 1          # sides (a, a, a+1)
        m = u + 2 * v
        pB = 3 * (m * m + v * v) - 1          # sides (a, a, a-1)
        if pA > LIMIT and pB > LIMIT:
            return total
        if pA <= LIMIT:
            total += pA
        if pB <= LIMIT:
            total += pB
        u, v = 2 * u + 3 * v, u + 2 * v


run(euler_94)

 94: Almost Equilateral Triangles                  0 msec ⇒ 518408346         ✅ 

## [Problem 95](https://projecteuler.net/problem=95)

*Numpy divisor-sum sieve, then walk chains, only crediting cycles at their minimum.*

In [97]:
def euler_95():
    """Amicable Chains"""
    N = 1_000_000
    s = divisor_sum_sieve(N).tolist()
    best_len, best_min = 0, 0
    for n in range(2, N + 1):
        chain_len, m = 1, s[n]
        while n < m <= N and chain_len <= 60:
            chain_len += 1
            m = s[m]
        if m == n and chain_len > best_len:   # a cycle whose minimum element is n
            best_len, best_min = chain_len, n
    return best_min


run(euler_95)

 95: Amicable Chains                             153 msec ⇒ 14316             ✅ 

## [Problem 96](https://projecteuler.net/problem=96)

*Bitmask backtracking with the minimum-remaining-values heuristic.*

In [98]:
def euler_96():
    """Su Doku"""
    def solve(board):
        rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
        for i, v in enumerate(board):
            if v:
                r, c = divmod(i, 9)
                bit = 1 << v
                rows[r] |= bit; cols[c] |= bit; boxes[r // 3 * 3 + c // 3] |= bit
        def backtrack():
            best_i, best_cand, best_count = -1, 0, 10
            for i in range(81):
                if board[i] == 0:
                    r, c = divmod(i, 9)
                    cand = ~(rows[r] | cols[c] | boxes[r // 3 * 3 + c // 3]) & 0b1111111110
                    cnt = cand.bit_count()
                    if cnt < best_count:
                        best_i, best_cand, best_count = i, cand, cnt
                        if cnt <= 1:
                            break
            if best_i == -1:
                return True
            r, c = divmod(best_i, 9)
            b = r // 3 * 3 + c // 3
            cand = best_cand
            while cand:
                bit = cand & -cand
                cand -= bit
                board[best_i] = bit.bit_length() - 1
                rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
                if backtrack():
                    return True
                rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
            board[best_i] = 0
            return False
        backtrack()
        return board
    lines = [line for line in DATA[96].splitlines() if not line.startswith('Grid')]
    total = 0
    for i in range(0, len(lines), 9):
        board = [int(ch) for row in lines[i:i + 9] for ch in row]
        solve(board)
        total += board[0] * 100 + board[1] * 10 + board[2]
    return total


run(euler_96)

 96: Su Doku                                      26 msec ⇒ 24702             ✅ 

## [Problem 97](https://projecteuler.net/problem=97)

*Modular exponentiation: the last ten digits never require the full number.*

In [99]:
def euler_97():
    """Large Non-Mersenne Prime"""
    M = 10 ** 10
    return (28433 * pow(2, 7830457, M) + 1) % M


run(euler_97)

 97: Large Non-Mersenne Prime                      0 msec ⇒ 8739992577        ✅ 

## [Problem 98](https://projecteuler.net/problem=98)

*Group words into anagram pairs; try mapping each onto squares of the right length.*

In [100]:
def euler_98():
    """Anagramic Squares"""
    words = DATA[98].replace('"', '').split(',')
    groups = defaultdict(list)
    for w in words:
        groups[''.join(sorted(w))].append(w)
    pairs = [(a, b) for g in groups.values() if len(g) > 1
             for a, b in combinations(g, 2)]
    lengths = {len(a) for a, b in pairs}
    squares_by_len = defaultdict(set)
    for k in count(1):
        sq = str(k * k)
        if len(sq) > max(lengths):
            break
        if len(sq) in lengths:
            squares_by_len[len(sq)].add(sq)
    best = 0
    for a, b in pairs:
        for sq in squares_by_len[len(a)]:
            mapping = {}
            if (len(set(sq)) == len(set(a))
                    and all(mapping.setdefault(ch, d) == d for ch, d in zip(a, sq))):
                t = ''.join(mapping[ch] for ch in b)
                if t[0] != '0' and t in squares_by_len[len(b)]:
                    best = max(best, int(sq), int(t))
    return best


run(euler_98)

 98: Anagramic Squares                            19 msec ⇒ 18769             ✅ 

## [Problem 99](https://projecteuler.net/problem=99)

*Compare b×log(a) instead of computing the towers.*

In [101]:
def euler_99():
    """Largest Exponential"""
    lines = DATA[99].splitlines()
    def key(i):
        a, b = map(int, lines[i].split(','))
        return b * log(a)
    return 1 + max(range(len(lines)), key=key)


run(euler_99)

 99: Largest Exponential                           0 msec ⇒ 709               ✅ 

## [Problem 100](https://projecteuler.net/problem=100)

*The recurrence for solutions of 2b(b-1) = n(n-1); jump straight past 10^12.*

In [102]:
def euler_100():
    """Arranged Probability"""
    b, n = 15, 21
    while n <= 10 ** 12:
        b, n = 3 * b + 2 * n - 2, 4 * b + 3 * n - 3
    return b


run(euler_100)

100: Arranged Probability                          0 msec ⇒ 756872327473      ✅ 

# Summary of Runs

In [103]:
runs()

Problems: 100
Run time in seconds: total: 1.7, max: 0.5, mean: 0.017, median: 0.001

  1: Multiples of 3 or 5                           0 msec ⇒ 233168            ✅ 
  2: Even Fibonacci Numbers                        0 msec ⇒ 4613732           ✅ 
  3: Largest Prime Factor                          0 msec ⇒ 6857              ✅ 
  4: Largest Palindrome Product                    0 msec ⇒ 906609            ✅ 
  5: Smallest Multiple                             0 msec ⇒ 232792560         ✅ 
  6: Sum Square Difference                         0 msec ⇒ 25164150          ✅ 
  7: 10001st Prime                                 1 msec ⇒ 104743            ✅ 
  8: Largest Product in a Series                   0 msec ⇒ 23514624000       ✅ 
  9: Special Pythagorean Triplet                   4 msec ⇒ 31875000          ✅ 
 10: Summation of Primes                           4 msec ⇒ 142913828922      ✅ 
 11: Largest Product in a Grid                     0 msec ⇒ 70600674          ✅ 
 12: Highly Divisible Tr